# 03 Data cleaning
**Chapter 3: Methodology — C5 Data cleaning, worked example.** The Python twin of `03_data_cleaning.do`.

Raw file: `ewallet_survey_raw.csv` (hypothetical survey, 414 records). Output: `ewallet_survey_clean.csv`.
Every step mirrors the do-file, so both produce the same 400 clean records.

In [1]:
# Setup: install anything missing (needed on Google Colab), then load the libraries.
import importlib.util, subprocess, sys
for pkg in ["pandas", "numpy", "scipy", "statsmodels", "linearmodels", "matplotlib", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Data live in ../data next to this notebook; online (e.g. Colab) they are read from GitHub.
DATA = Path("../data")
RAW_URL = "https://raw.githubusercontent.com/soonjanjan-lang/Chapter3-Methodology-lectnotes-dashboard/main/downloads/data/"

def load(name):
    p = DATA / name
    return pd.read_csv(p) if p.exists() else pd.read_csv(RAW_URL + name)

## 1. Inspect structure

In [2]:
raw = load("ewallet_survey_raw.csv")
print("Raw records:", len(raw))
raw.info()

Raw records: 414
<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   resp_id           414 non-null    int64  
 1   faculty           414 non-null    str    
 2   year_study        414 non-null    int64  
 3   gender            414 non-null    str    
 4   age               414 non-null    int64  
 5   state             414 non-null    str    
 6   hometown_urban    414 non-null    int64  
 7   allowance_rm      414 non-null    float64
 8   ewallet_user      414 non-null    int64  
 9   ewallet_days      414 non-null    int64  
 10  ewallet_spend_rm  414 non-null    float64
 11  PU1               406 non-null    float64
 12  PU2               406 non-null    float64
 13  PU3               406 non-null    float64
 14  PU4               406 non-null    float64
 15  PS1               406 non-null    float64
 16  PS2               406 non-null    floa

In [3]:
print(raw["gender"].value_counts(dropna=False).head(12))
raw[["age", "allowance_rm", "ewallet_days", "PU1", "PS1", "EOU1", "intention"]].describe().round(2)

gender
Female     164
Male       110
F           18
female      18
FEMALE      16
Female      15
Male        14
m           12
M           12
male        12
MALE        12
f           11
Name: count, dtype: int64


,age,allowance_rm,ewallet_days,PU1,PS1,EOU1,intention
count,414.00,414.00,414.00,406.00,406.00,406.00,406.00
mean,28.05,1247.38,2.35,3.04,3.14,3.10,3.15
std,83.31,5235.61,2.18,1.72,1.26,1.36,1.43
min,18.00,-9.00,0.00,-9.00,1.00,-9.00,1.00
25%,20.00,590.00,0.00,2.00,2.00,2.00,2.00
50%,21.00,825.00,2.00,3.00,3.00,3.00,3.00
75%,22.00,1140.00,4.00,4.00,4.00,4.00,4.00
max,999.00,98000.00,10.00,5.00,6.00,6.00,5.00


## 2. Duplicates

In [4]:
print("Repeated IDs:", raw["resp_id"].duplicated().sum())
print("Rows identical on every variable:", raw.duplicated().sum())
df = raw.drop_duplicates().copy()
print("After dropping exact duplicates:", len(df))

Repeated IDs: 6
Rows identical on every variable: 6
After dropping exact duplicates: 408


## 3. Incomplete questionnaires (break-offs)
Drop anyone who answered fewer than half of the 15 scale items.

In [5]:
items = [f"PU{j}" for j in range(1, 5)] + [f"PS{j}" for j in range(1, 5)] + [f"EOU{j}" for j in range(1, 4)]
scale = items + ["intention", "finlit1", "finlit2", "finlit3"]
n_missing = df[scale].isna().sum(axis=1)
print(n_missing[n_missing > 0].value_counts())
df = df[n_missing < 8].copy()
print("After dropping break-offs:", len(df))

15    8
Name: count, dtype: int64
After dropping break-offs: 400


## 4. Missing-value codes and 5. out-of-range or implausible values

In [6]:
df["age"] = df["age"].replace(999, np.nan)
for v in ["allowance_rm"] + items + ["intention"]:
    df[v] = df[v].replace(-9, np.nan)
for v in items + ["intention"]:
    bad = df[v].notna() & ~df[v].between(1, 5)
    if bad.any():
        print(f"{v}: {bad.sum()} out-of-range value(s) set to missing")
    df.loc[bad, v] = np.nan
print(df.loc[df["age"].notna() & ~df["age"].between(17, 40), ["resp_id", "age"]])
df.loc[~df["age"].between(17, 40), "age"] = np.nan
print(df.loc[df["allowance_rm"] > 10000, ["resp_id", "allowance_rm"]])
df.loc[df["allowance_rm"] > 10000, "allowance_rm"] = np.nan

PU2: 1 out-of-range value(s) set to missing
PU4: 1 out-of-range value(s) set to missing
PS1: 1 out-of-range value(s) set to missing
PS3: 1 out-of-range value(s) set to missing
EOU1: 1 out-of-range value(s) set to missing
     resp_id    age
191     1062  150.0
     resp_id  allowance_rm
10      1121       98000.0
324     1310       44000.0


## 6. Inconsistent text coding

In [7]:
g = df["gender"].str.strip().str.lower()
df["female"] = np.select([g.isin(["f", "female"]), g.isin(["m", "male"])], [1, 0], default=np.nan)
print(pd.crosstab(df["gender"], df["female"]))
df["state"] = df["state"].str.strip().str.title()
print(df["state"].value_counts())

female   0.0  1.0
gender           
F          0   17
FEMALE     0   16
Female     0  158
Female     0   14
M         12    0
MALE      11    0
Male     108    0
Male      13    0
f          0   11
female     0   17
m         12    0
male      11    0
state
Selangor           52
Kedah              47
Perak              39
Johor              38
Sarawak            35
Kelantan           34
Pulau Pinang       32
Sabah              25
Perlis             23
Pahang             22
Melaka             18
Terengganu         14
Negeri Sembilan    14
Kuala Lumpur        7
Name: count, dtype: int64


## 7. Logical consistency (skip pattern)

In [8]:
print("Days > 7:", (df["ewallet_days"] > 7).sum())
df.loc[df["ewallet_days"] > 7, "ewallet_days"] = np.nan
contra = (df["ewallet_user"] == 0) & (df["ewallet_days"] > 0)
print("Non-users reporting use:", contra.sum())
df.loc[contra, "ewallet_days"] = np.nan

Days > 7: 3
Non-users reporting use: 4


## 8. Generate analysis variables

In [9]:
df["PU"] = df[[f"PU{j}" for j in range(1, 5)]].mean(axis=1)
df["PS"] = df[[f"PS{j}" for j in range(1, 5)]].mean(axis=1)
df["EOU"] = df[[f"EOU{j}" for j in range(1, 4)]].mean(axis=1)
df["finlit"] = df[["finlit1", "finlit2", "finlit3"]].sum(axis=1, min_count=1)
df["ln_allow"] = np.log(df["allowance_rm"])
df["urban"] = df["hometown_urban"]

## 9. Save raw and clean separately (never overwrite the raw file)

In [10]:
order = ["resp_id", "faculty", "year_study", "female", "age", "state", "urban", "allowance_rm", "ln_allow",
         "ewallet_user", "ewallet_days", "ewallet_spend_rm", "PU", "PS", "EOU", "finlit", "intention", "satisfaction"]
clean = df[order + [c for c in df.columns if c not in order and c not in ("gender", "hometown_urban")]]
print("Clean records:", len(clean))
print(clean[["female", "age", "allowance_rm", "ewallet_user", "PU", "PS", "EOU", "finlit"]].describe().T.round(4))
if DATA.exists():
    clean.to_csv(DATA / "ewallet_survey_clean.csv", index=False)

Clean records: 400
              count      mean       std    min       25%       50%     75%     max
female        400.0    0.5825    0.4938    0.0    0.0000    1.0000     1.0     1.0
age           396.0   20.6616    1.4948   18.0   20.0000   21.0000    22.0    24.0
allowance_rm  394.0  912.2843  444.1527  150.0  592.5000  820.0000  1130.0  3280.0
ewallet_user  400.0    0.7525    0.4321    0.0    1.0000    1.0000     1.0     1.0
PU            400.0    3.1433    1.0124    1.0    2.5000    3.2500     4.0     5.0
PS            400.0    3.1431    1.0218    1.0    2.5000    3.2500     4.0     5.0
EOU           400.0    3.1533    0.9816    1.0    2.3333    3.1667     4.0     5.0
finlit        400.0    1.8975    0.9997    0.0    1.0000    2.0000     3.0     3.0


## 10. A data dictionary for the appendix

In [11]:
# a data dictionary you can export to Word or Excel for the appendix
codebook = pd.DataFrame({
    "variable": ["ewallet_user", "PU", "finlit", "female"],
    "definition": ["Used an e-wallet in past 30 days", "Perceived usefulness, mean of PU1-PU4",
                   "Financial literacy, correct answers", "Respondent is female"],
    "level": ["nominal (binary)", "interval-like", "interval-like", "nominal (binary)"],
    "coding": ["1 = yes, 0 = no", "1-5", "0-3", "1 = female, 0 = male"],
})
codebook.to_excel(DATA / "codebook_ewallet.xlsx" if DATA.exists() else "codebook.xlsx", index=False)
clean[codebook["variable"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ewallet_user,400.0,0.752500,0.432100,0.0,1.0,1.00,1.0,1.0
PU,400.0,3.143333,1.012408,1.0,2.5,3.25,4.0,5.0
finlit,400.0,1.897500,0.999746,0.0,1.0,2.00,3.0,3.0
female,400.0,0.582500,0.493764,0.0,0.0,1.00,1.0,1.0


**Check against Stata:** 400 clean records; mean of `female` 0.5825, `ewallet_user` 0.7525, `PU` 3.1433, `finlit` 1.8975 —
identical to the summary at the end of `03_data_cleaning.log`.

In [12]:
assert len(clean) == 400
assert round(clean["female"].mean(), 4) == 0.5825 and round(clean["ewallet_user"].mean(), 4) == 0.7525
assert round(clean["PU"].mean(), 4) == 3.1433 and round(clean["finlit"].mean(), 4) == 1.8975
print("Matches the Stata results.")

Matches the Stata results.
